# Dubins car — Vanilla MPC robustness under additive position disturbances

The first section is identical to the original notebook (nominal, no disturbance) with full trajectory simulation.
The second section adds a position-only additive disturbance to the plant and sweeps over levels.

**Disturbance model (plant-side only, controller stays nominal):**
```
ẋ₁ = v·cos(θ) + d₁,   ẋ₂ = v·sin(θ) + d₂,   θ̇ = u₁,   v̇ = u₂
d₁, d₂ ~ Uniform[-ε, ε],  resampled each MPC step
```
Disturbance applied **only** to position channels (ẋ₁, ẋ₂). Heading and speed are
controlled perfectly by the MPC — exactly as in the nominal case.

**ε is scaled relative to typical position drift v·cos(θ) ≈ 0.5 m/s:**

| Level    | ε (m/s) | fraction of drift |
|----------|:-------:|:-----------------:|
| Nominal  | 0.00    | 0%                |
| Mild     | 0.10    | ~20%              |
| Moderate | 0.30    | ~60%              |
| Strong   | 0.50    | ~100%             |

In [ ]:
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, ROOT)

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
import casadi as ca

# ── Symbolic variables ────────────────────────────────────────────────────────
x1, x2, th, v = sp.symbols('x1 x2 th v')
y1, y2 = sp.symbols('y1 y2')
state_vars = [x1, x2, th, v]

# ── Dubins-car dynamics:  xdot = f(x) + g(x)*u ───────────────────────────────
f_sym = sp.Matrix([v * sp.cos(th), v * sp.sin(th), 0, 0])
g_sym = sp.Matrix([[0, 0], [0, 0], [1, 0], [0, 1]])
hx    = sp.Matrix([x1, x2])

# ── Sets in output (y) space ─────────────────────────────────────────────────
h_raw        = -(y1**4 + y2**4 - 16) * (y1**4 + y2**4 - 4)
target_set_y = (y2 - 0)**2 + (2*(y1 + 1.7))**2 - 0.4
alpha        = 1e-3 * (-target_set_y + 300)
safe_set_y   = alpha * h_raw

safe_set_x   = safe_set_y.subs({y1: hx[0], y2: hx[1]})
target_set_x = target_set_y.subs({y1: hx[0], y2: hx[1]})

safe_set_y_func   = sp.lambdify([y1, y2], safe_set_y, 'numpy')
target_set_y_func = sp.lambdify([y1, y2], target_set_y, 'numpy')
safe_set_x_func   = sp.lambdify(state_vars, safe_set_x, 'numpy')
target_set_x_func = sp.lambdify(state_vars, target_set_x, 'numpy')
hx_func           = sp.lambdify(state_vars, hx, 'numpy')

print('safe_set   (y):', safe_set_y)
print('target_set (y):', target_set_y)

In [ ]:
# ── CasADi MPC formulation ── IDENTICAL TO ORIGINAL ──────────────────────────
# max_iter set to 600 to match sampling_sweep notebook (consistent solver settings)
dt_mpc  = 0.05
N_hor   = 25
u1_max  = 5.0
u2_max  = 5.0
TARGET_POS = np.array([-1.7, 0.0])

xc = ca.MX.sym('x', 4)
uc = ca.MX.sym('u', 2)
fc = ca.vertcat(xc[3]*ca.cos(xc[2]), xc[3]*ca.sin(xc[2]), uc[0], uc[1])
rhs_ca = ca.Function('f', [xc, uc], [fc])

def rk4_ca(x, u):
    k1 = rhs_ca(x, u)
    k2 = rhs_ca(x + dt_mpc/2*k1, u)
    k3 = rhs_ca(x + dt_mpc/2*k2, u)
    k4 = rhs_ca(x + dt_mpc*k3, u)
    return x + (dt_mpc/6)*(k1 + 2*k2 + 2*k3 + k4)

def safe_ca(x):
    y1v, y2v = x[0], x[1]
    h   = -(y1v**4 + y2v**4 - 16)*(y1v**4 + y2v**4 - 4)
    tgt = y2v**2 + (2*(y1v + 1.7))**2 - 0.4
    alp = 1e-3*(-tgt + 300)
    return alp * h

def target_ca(x):
    y1v, y2v = x[0], x[1]
    return y2v**2 + (2*(y1v + 1.7))**2 - 0.4

def h_ca(xv):
    return ca.vertcat(xv[0], xv[1])

_IPOPT_OPTS = {
    'ipopt.print_level': 0, 'print_time': 0,
    'ipopt.max_iter': 600,    # standardised to 600 across all experiment notebooks
    'ipopt.tol': 1e-4, 'ipopt.acceptable_tol': 1e-3,
}

def solve_mpc(x0_val, U_warm=None):
    opti = ca.Opti()
    X = opti.variable(4, N_hor + 1)
    U = opti.variable(2, N_hor)
    x0_np = np.asarray(x0_val, dtype=float).flatten()
    opti.subject_to(X[:, 0] == x0_np)
    for k in range(N_hor):
        opti.subject_to(X[:, k+1] == rk4_ca(X[:, k], U[:, k]))
        opti.subject_to(safe_ca(X[:, k]) >= 0)
    opti.subject_to(safe_ca(X[:, N_hor]) >= 0)
    opti.subject_to(target_ca(X[:, N_hor]) <= 0)
    opti.subject_to(opti.bounded(-u1_max, U[0, :], u1_max))
    opti.subject_to(opti.bounded(-u2_max, U[1, :], u2_max))
    tgt  = ca.DM(TARGET_POS)
    Q_y  = ca.DM([[5.0, 0], [0, 5.0]])
    Qf_y = ca.DM([[80.0, 0], [0, 80.0]])
    R_u  = ca.DM([[0.05, 0], [0, 0.05]])
    cost = 0
    for k in range(N_hor):
        dy    = h_ca(X[:, k]) - tgt
        cost += dy.T @ Q_y @ dy + U[:, k].T @ R_u @ U[:, k]
    dy_N  = h_ca(X[:, N_hor]) - tgt
    cost += dy_N.T @ Qf_y @ dy_N
    opti.minimize(cost)
    if U_warm is not None:
        U_ws = np.hstack([U_warm[:, 1:], U_warm[:, -1:]])
        X_ws = np.zeros((4, N_hor+1)); X_ws[:, 0] = x0_np
        for k in range(N_hor):
            xk, uk = X_ws[:, k], U_ws[:, k]
            X_ws[:, k+1] = xk + dt_mpc*np.array(
                [xk[3]*np.cos(xk[2]), xk[3]*np.sin(xk[2]), uk[0], uk[1]])
        opti.set_initial(X, X_ws); opti.set_initial(U, U_ws)
    else:
        tgt_full = np.array([TARGET_POS[0], TARGET_POS[1], x0_np[2], 0.0])
        X_ws = np.array([(1-k/N_hor)*x0_np + (k/N_hor)*tgt_full
                         for k in range(N_hor+1)]).T
        opti.set_initial(X, X_ws); opti.set_initial(U, 0)
    opti.solver('ipopt', _IPOPT_OPTS)
    try:
        sol = opti.solve()
        return True, sol.value(U)
    except Exception:
        return False, None

print(f'MPC ready: N_hor={N_hor}, dt={dt_mpc} s, u_bounds=[±{u1_max}, ±{u2_max}]')
print('Terminal constraint: target_set(x_N) <= 0')
print('max_iter=600 (standardised across all experiment notebooks)')

In [ ]:
# ── Sample the 4-D state space (same seed & bounds as original notebook) ──────
from scipy.integrate import solve_ivp
from joblib import Parallel, delayed

N_samples = 1000
np.random.seed(42)

lower_bound = np.array([-2.0, -2.0, 2*np.pi/3, -1.0])
upper_bound = np.array([ 2.0,  2.0, 4*np.pi/3,  1.0])

x_samples = np.random.rand(4, N_samples) * (upper_bound - lower_bound).reshape(-1, 1) \
            + lower_bound.reshape(-1, 1)

safe_vals   = np.atleast_1d(np.squeeze(safe_set_x_func(*x_samples)))
target_vals = np.atleast_1d(np.squeeze(target_set_x_func(*x_samples)))

candidate_mask = (safe_vals >= 0) & (target_vals > 0)
candidate_idx  = np.where(candidate_mask)[0]
candidate_idx  = candidate_idx[:300]
print(f'Total samples : {N_samples}')
print(f'Candidates (safe & outside target): {len(candidate_idx)}')

def _check_one(idx):
    ok, _ = solve_mpc(x_samples[:, idx])
    return idx, ok

print(f'\nChecking MPC feasibility ({len(candidate_idx)} candidates, '
      f'N_hor={N_hor}, dt={dt_mpc}s) ...')

results = Parallel(n_jobs=-1, prefer='processes', verbose=5)(
    delayed(_check_one)(idx) for idx in candidate_idx
)

mpc_feasible_idx   = np.array([idx for idx, ok in results if ok])
mpc_infeasible_idx = np.array([idx for idx, ok in results if not ok])

n_c = len(candidate_idx)
print(f'\nMPC feasible:   {len(mpc_feasible_idx):3d} / {n_c}  '
      f'({100*len(mpc_feasible_idx)/n_c:.1f}%)')
print(f'MPC infeasible: {len(mpc_infeasible_idx):3d} / {n_c}  '
      f'({100*len(mpc_infeasible_idx)/n_c:.1f}%)')

In [ ]:
# ── Visualise MPC-feasible / MPC-infeasible initial states ── ORIGINAL PLOT ──
y_feasible   = np.array(hx_func(*x_samples[:, mpc_feasible_idx])).reshape(2, -1)
y_infeasible = np.array(hx_func(*x_samples[:, mpc_infeasible_idx])).reshape(2, -1)

n_grid = 400
y1_g = np.linspace(-2.1, 2.1, n_grid)
y2_g = np.linspace(-2.1, 2.1, n_grid)
Y1, Y2 = np.meshgrid(y1_g, y2_g)
Z_safe   = safe_set_y_func(Y1, Y2)
Z_target = target_set_y_func(Y1, Y2)

px_fig = 1 / plt.rcParams['figure.dpi']
fig, ax = plt.subplots(figsize=(650*px_fig, 600*px_fig), layout='constrained')
fig.set_dpi(200)

ax.contourf(Y1, Y2, Z_safe,   levels=[-np.inf, 0], colors=['#ffcccc'], alpha=0.6, zorder=1)
ax.contour( Y1, Y2, Z_safe,   levels=[0], colors=['#b2182b'], linewidths=2, zorder=2)
ax.contourf(Y1, Y2, Z_target, levels=[-np.inf, 0], colors=['#fee0d2'], alpha=1.0, zorder=3)
ax.contour( Y1, Y2, Z_target, levels=[0], colors=['#d6604d'], linewidths=3, zorder=4)

ax.scatter(y_infeasible[0], y_infeasible[1], s=12, color='#d73027', alpha=0.65,
           marker='x', linewidths=0.9, label=f'MPC infeasible: {len(mpc_infeasible_idx)}', zorder=9)
ax.scatter(y_feasible[0],   y_feasible[1],   s=12, color='#2166ac', alpha=0.65,
           marker='o', label=f'MPC feasible: {len(mpc_feasible_idx)}', zorder=10)

ax.set_xlabel('$y_1 = x_1$ [m]', fontsize=16)
ax.set_ylabel('$y_2 = x_2$ [m]', fontsize=16)
ax.set_title('MPC feasibility — sampled initial states in position space', fontsize=13)
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ── NOMINAL receding-horizon MPC simulation ── IDENTICAL TO ORIGINAL ──────────
T_max   = 10.0
phi_tol = 0.05

def simulate_mpc_traj(x0):
    """Simulate one trajectory under receding-horizon MPC (nominal plant)."""
    x = np.asarray(x0, dtype=float).copy()
    xs, ts = [x.copy()], [0.0]
    t = 0.0
    U_prev = None

    while t < T_max - 1e-9:
        if float(np.squeeze(target_set_x_func(*x))) <= 0:
            break
        ok, U_opt = solve_mpc(x, U_warm=U_prev)
        if not ok:
            break
        u0     = np.array(U_opt[:, 0]).flatten()
        U_prev = U_opt

        def rhs(_, x_):
            return [x_[3]*np.cos(x_[2]), x_[3]*np.sin(x_[2]),
                    float(u0[0]), float(u0[1])]

        sol = solve_ivp(rhs, [t, t+dt_mpc], x, method='DOP853',
                        rtol=1e-8, atol=1e-10, max_step=dt_mpc/10)
        if not np.all(np.isfinite(sol.y)):
            break
        x  = sol.y[:, -1]; t += dt_mpc
        xs.append(x.copy()); ts.append(t)

    ts = np.array(ts); xs = np.array(xs)
    ys = np.array(hx_func(*xs.T)).reshape(2, -1).T
    return ts, xs, ys

# Use ALL MPC-feasible ICs (not just 50) — consistent with paper's 300-candidate analysis
N_sim_try = len(mpc_feasible_idx)   # up to 245 for vanilla MPC
x_sim     = x_samples[:, mpc_feasible_idx]

print(f'Simulating {N_sim_try} nominal trajectories (all MPC-feasible ICs) ...')
raw = []
for i in range(N_sim_try):
    print(f'  [{i+1:3d}/{N_sim_try}] ', end='', flush=True)
    res = simulate_mpc_traj(x_sim[:, i])
    raw.append(res)
    tt, tx, _ = res
    phi_f = float(np.squeeze(target_set_x_func(*tx[-1])))
    print(f'T={tt[-1]:.2f}s  {"REACHED" if phi_f<=0 else f"phi={phi_f:.4f}"}')

traj_t = [r[0] for r in raw]
traj_x = [r[1] for r in raw]
traj_y = [r[2] for r in raw]

n_nom_ok = sum(1 for tt, tx, _ in zip(traj_t, traj_x, traj_y)
               if float(np.squeeze(target_set_x_func(*tx[-1]))) <= phi_tol)
print(f'\nNominal success: {n_nom_ok}/{N_sim_try} ({100*n_nom_ok/N_sim_try:.1f}%)')

In [ ]:
# ── Filter + visualise nominal closed-loop trajectories ── ORIGINAL PLOT ──────
from functional import BetterColor

T_min = 1.1

filtered = [
    (tt, tx, ty)
    for tt, tx, ty in zip(traj_t, traj_x, traj_y)
    if tt[-1] < T_max - 1e-6
    and tt[-1] >= T_min
    and float(np.squeeze(target_set_x_func(*tx[-1]))) <= phi_tol
][1:2]

print(f'Kept {len(filtered)}/{len(traj_t)} trajectories for nominal plot')
traj_t_plot, traj_x_plot, traj_y_plot = zip(*filtered) if filtered else ([], [], [])

n_grid = 400
y1_g = np.linspace(-2.1, 2.1, n_grid); y2_g = np.linspace(-2.1, 2.1, n_grid)
Y1, Y2 = np.meshgrid(y1_g, y2_g)
Z_safe_y   = safe_set_y_func(Y1, Y2)
Z_target_y = target_set_y_func(Y1, Y2)

px_fig = 1 / plt.rcParams['figure.dpi']
fig, ax = plt.subplots(figsize=(650*px_fig, 600*px_fig), layout='constrained')
fig.set_dpi(200)

ax.contourf(Y1, Y2, Z_safe_y,   levels=[0, np.inf],  colors=['#e6f2ff'], alpha=0.3, zorder=1)
ax.contour( Y1, Y2, Z_safe_y,   levels=[0], colors=['#2166ac'], linewidths=2, zorder=2)
ax.contourf(Y1, Y2, Z_target_y, levels=[-np.inf, 0], colors=['#fee0d2'], alpha=1.0, zorder=3)
ax.contour( Y1, Y2, Z_target_y, levels=[0], colors=['#d6604d'], linewidths=2, zorder=4)

ax.scatter(y_infeasible[0], y_infeasible[1], s=50, color='#d73027', alpha=0.85,
           marker='x', linewidths=3, label=f'MPC infeasible: {len(mpc_infeasible_idx)}', zorder=6)
ax.scatter(y_feasible[0],   y_feasible[1],   s=50, color='#2166ac', alpha=0.35,
           marker='o', label=f'MPC feasible: {len(mpc_feasible_idx)}', zorder=7)

for ty in traj_y_plot:
    ax.plot(ty[:, 0], ty[:, 1], color='black', lw=3, alpha=0.65, zorder=10, linestyle='--')
    ax.scatter(ty[0, 0],  ty[0, 1],  s=45, c=[BetterColor.orange3()],
               marker='o', linewidths=0.4, zorder=20)
    ax.scatter(ty[-1, 0], ty[-1, 1], s=45, c=[BetterColor.green0()],
               marker='x', linewidths=3.0, zorder=20)

fs = 30
ax.set_xlabel('$y_1 = x_1$', fontsize=fs-4)
ax.set_ylabel('$y_2 = x_2$', fontsize=fs-4)
ax.xaxis.set_tick_params(labelsize=25)
ax.yaxis.set_tick_params(labelsize=25)
ax.set_title('Nominal (\u03b5=0) vanilla MPC', fontsize=13)
plt.savefig('fig_nominal_vanilla_mpc.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Control inputs along nominal trajectory ── ORIGINAL PLOT ──────────────────
px_fig = 1 / plt.rcParams['figure.dpi']
fig, axes = plt.subplots(2, 1, figsize=(700*px_fig, 500*px_fig),
                         layout='constrained', sharex=True)
fig.set_dpi(200)

for tt, tx in zip(traj_t_plot, traj_x_plot):
    if len(tt) < 2: continue
    dt_arr   = np.diff(tt)
    u1_approx = np.diff(tx[:, 2]) / dt_arr
    u2_approx = np.diff(tx[:, 3]) / dt_arr
    axes[0].step(tt[:-1], u1_approx, lw=1.0, alpha=0.7, where='post')
    axes[1].step(tt[:-1], u2_approx, lw=1.0, alpha=0.7, where='post')

for ax, lbl, bnd in zip(axes,
    ['$u_1$ (angular vel.) [rad/s]', '$u_2$ (acceleration) [m/s\u00b2]'],
    [u1_max, u2_max]):
    ax.axhline( bnd, color='gray', lw=1.0, ls='--', alpha=0.6, label=f'bound \u00b1{bnd}')
    ax.axhline(-bnd, color='gray', lw=1.0, ls='--', alpha=0.6)
    ax.axhline(0,    color='gray', lw=0.8, ls=':')
    ax.set_ylabel(lbl, fontsize=14)
    ax.tick_params(labelsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9, loc='upper right')

axes[1].set_xlabel('Time [s]', fontsize=14)
fig.suptitle('Control inputs — nominal vanilla MPC', fontsize=13)
plt.show()

---
## Robustness Study — Additive Position Disturbance

The cells below extend the nominal experiment with position disturbances.
The MPC formulation and controller are **unchanged**. Only the plant's `rhs` adds `d₁, d₂`.

In [ ]:
import pandas as pd

# ── Disturbance levels ─────────────────────────────────────────────────────────
# eps is in m/s; applied to position-derivative channels (ẋ₁, ẋ₂) only.
# Typical position drift ≈ v·cos(θ) ≈ 0.5 m/s.
LEVELS = [
    {'name': 'nominal',  'eps': 0.00},
    {'name': 'mild',     'eps': 0.10},
    {'name': 'moderate', 'eps': 0.30},
    {'name': 'strong',   'eps': 0.50},
]
DIST_SEED = 123

def simulate_mpc_traj_perturbed(x0, eps_pos, rng):
    """
    Vanilla MPC on perturbed plant.
    ONLY change vs. simulate_mpc_traj: two disturbance terms d₁, d₂
    added to the position-derivative channels. Heading and speed are exact.

    Returns (ts, xs, ys, metrics_dict).
    """
    x = np.asarray(x0, dtype=float).copy()
    xs, ts = [x.copy()], [0.0]
    t = 0.0
    U_prev = None
    n_infeasible   = 0
    control_effort = 0.0

    while t < T_max - 1e-9:
        if float(np.squeeze(target_set_x_func(*x))) <= 0:
            break

        # MPC uses current (disturbance-affected) state measurement
        ok, U_opt = solve_mpc(x, U_warm=U_prev)
        if not ok:
            n_infeasible += 1
            break
        u0     = np.array(U_opt[:, 0]).flatten()
        U_prev = U_opt

        # Disturbance on position channels only (resampled each MPC step)
        d = rng.uniform(-eps_pos, eps_pos, size=2) if eps_pos > 0 else np.zeros(2)

        # ── PERTURBED plant rhs ── only these two lines differ from nominal ──
        def rhs(_, x_):
            return [
                x_[3]*np.cos(x_[2]) + d[0],   # ← disturbance on ẋ₁
                x_[3]*np.sin(x_[2]) + d[1],   # ← disturbance on ẋ₂
                float(u0[0]),                  # heading: exact (no disturbance)
                float(u0[1]),                  # speed:   exact (no disturbance)
            ]

        sol = solve_ivp(rhs, [t, t+dt_mpc], x, method='DOP853',
                        rtol=1e-8, atol=1e-10, max_step=dt_mpc/10)
        if not np.all(np.isfinite(sol.y)):
            break

        x  = sol.y[:, -1]; t += dt_mpc
        xs.append(x.copy()); ts.append(t)
        control_effort += (u0[0]**2 + u0[1]**2) * dt_mpc

    ts = np.array(ts); xs = np.array(xs)
    ys = np.array(hx_func(*xs.T)).reshape(2, -1).T

    phi_f   = float(np.squeeze(target_set_x_func(*xs[-1])))
    success = phi_f <= phi_tol

    # Safety margin along trajectory
    psi_vals = np.array([float(np.squeeze(safe_set_x_func(*xs[i]))) for i in range(len(xs))])

    metrics = {
        'success':              success,
        'time_to_target':       float(ts[-1]) if success else float('nan'),
        'sim_duration':         float(ts[-1]),
        'min_safety_margin':    float(psi_vals.min()),
        'n_safety_violations':  int((psi_vals < 0).sum()),
        'control_effort':       control_effort,
        'mpc_infeasible_steps': n_infeasible,
        'n_steps':              len(ts) - 1,
    }
    return ts, xs, ys, metrics

print('Disturbance levels:')
for lv in LEVELS:
    print(f"  {lv['name']:10s}  eps={lv['eps']:.2f} m/s")

In [ ]:
# ── Run all disturbance levels (same 50 ICs as nominal) ───────────────────────
all_rows      = []
all_level_traj = {}   # store (ts, xs, ys) per level for plotting

for lv in LEVELS:
    level_name = lv['name']
    eps        = lv['eps']
    rng        = np.random.default_rng(DIST_SEED)

    print(f'\n=== Level: {level_name}  (eps={eps:.2f} m/s) ===')
    raw_lv = []
    for i in range(N_sim_try):
        print(f'  [{i+1:2d}/{N_sim_try}] ', end='', flush=True)
        ts_i, xs_i, ys_i, m = simulate_mpc_traj_perturbed(x_sim[:, i], eps, rng)
        raw_lv.append((ts_i, xs_i, ys_i))
        phi_f = float(np.squeeze(target_set_x_func(*xs_i[-1])))
        print(f'T={ts_i[-1]:.2f}s  '
              f'{"REACHED" if phi_f<=0 else f"phi={phi_f:.4f}"}  '
              f'min_psi={m["min_safety_margin"]:.4f}  '
              f'infeas={m["mpc_infeasible_steps"]}')
        m.update({'level': level_name, 'ic_idx': int(mpc_feasible_idx[i]), 'traj_id': i})
        all_rows.append(m)

    n_ok = sum(r['success'] for r in all_rows if r['level'] == level_name)
    print(f'  --> Success: {n_ok}/{N_sim_try} ({100*n_ok/N_sim_try:.1f}%)')
    all_level_traj[level_name] = raw_lv

df = pd.DataFrame(all_rows)
print('\nAll levels done.')
level_order = ['nominal', 'mild', 'moderate', 'strong']
print(df.groupby('level')[['success','time_to_target','min_safety_margin',
                             'n_safety_violations','mpc_infeasible_steps']]
        .mean().reindex(level_order).round(3))

In [ ]:
# ── Phase portraits for each disturbance level ─────────────────────────────────
# Each panel: feasibility scatter (blue/red) + up to 5 trajectory samples (black dashed)
# This matches the original paper figure style.
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('Vanilla MPC — Robustness under position disturbance (ε applied to ẋ₁, ẋ₂ only)',
             fontsize=13)

n_grid = 400
y1_g = np.linspace(-2.1, 2.1, n_grid); y2_g = np.linspace(-2.1, 2.1, n_grid)
Y1, Y2 = np.meshgrid(y1_g, y2_g)
Z_safe_y   = safe_set_y_func(Y1, Y2)
Z_target_y = target_set_y_func(Y1, Y2)

# Precompute position-space projections of feasible/infeasible ICs (same for all panels)
y_feas_plot   = np.array(hx_func(*x_samples[:, mpc_feasible_idx])).reshape(2, -1)
y_infeas_plot = np.array(hx_func(*x_samples[:, mpc_infeasible_idx])).reshape(2, -1)

for ax, level_name in zip(axes.flat, level_order):
    eps_label = next(lv['eps'] for lv in LEVELS if lv['name'] == level_name)

    # Set boundaries
    ax.contourf(Y1, Y2, Z_safe_y,   levels=[0, np.inf],  colors=['#e6f2ff'], alpha=0.3, zorder=1)
    ax.contour( Y1, Y2, Z_safe_y,   levels=[0], colors=['#2166ac'], linewidths=1.5, zorder=2)
    ax.contourf(Y1, Y2, Z_target_y, levels=[-np.inf, 0], colors=['#fee0d2'], alpha=1.0, zorder=3)
    ax.contour( Y1, Y2, Z_target_y, levels=[0], colors=['#d6604d'], linewidths=1.5, zorder=4)

    # Feasibility scatter (same style as original paper figure)
    ax.scatter(y_infeas_plot[0], y_infeas_plot[1], s=15, color='#d73027', alpha=0.6,
               marker='x', linewidths=0.8,
               label=f'MPC infeasible: {len(mpc_infeasible_idx)}', zorder=6)
    ax.scatter(y_feas_plot[0],  y_feas_plot[1],  s=15, color='#2166ac', alpha=0.35,
               marker='o',
               label=f'MPC feasible: {len(mpc_feasible_idx)}', zorder=7)

    # Successful trajectories for this level (show up to 5)
    raw_lv = all_level_traj[level_name]
    shown  = 0
    for ts_i, xs_i, ys_i in raw_lv:
        phi_f = float(np.squeeze(target_set_x_func(*xs_i[-1])))
        if phi_f <= phi_tol and ts_i[-1] >= T_min and ts_i[-1] < T_max - 1e-6:
            ax.plot(ys_i[:, 0], ys_i[:, 1], 'k--', lw=2.0, alpha=0.75, zorder=10)
            ax.scatter(ys_i[0, 0],  ys_i[0, 1],  s=40, c=[BetterColor.orange3()],
                       marker='o', zorder=20)
            ax.scatter(ys_i[-1, 0], ys_i[-1, 1], s=40, c=[BetterColor.green0()],
                       marker='x', linewidths=2.5, zorder=20)
            shown += 1
            if shown >= 5: break

    n_ok = sum(1 for r in all_rows if r['level'] == level_name and r['success'])
    n_tot = sum(1 for r in all_rows if r['level'] == level_name)
    ax.set_title(f'{level_name}  (ε={eps_label:.2f} m/s)  '
                 f'success {n_ok}/{n_tot} ({100*n_ok/n_tot:.0f}%)', fontsize=11)
    ax.set_xlabel('$y_1 = x_1$', fontsize=12)
    ax.set_ylabel('$y_2 = x_2$', fontsize=12)
    ax.grid(True, alpha=0.3)
    if level_name == 'nominal':
        ax.legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.savefig('fig_robustness_phase_vanilla_mpc.png', dpi=150, bbox_inches='tight')
plt.show()
print('Phase portrait grid saved.')

In [ ]:
# ── Save results ──────────────────────────────────────────────────────────────
df.to_csv('robustness_results_vanilla_mpc.csv', index=False)
print(f'Saved robustness_results_vanilla_mpc.csv  ({len(df)} rows)')

# ── Summary bar chart ─────────────────────────────────────────────────────────
summary = df.groupby('level').agg(
    success_rate   =('success',            lambda x: 100*x.mean()),
    mean_min_psi   =('min_safety_margin',  'mean'),
    mean_violations=('n_safety_violations','mean'),
    mean_effort    =('control_effort',     'mean'),
).reindex(level_order)

colors = ['#2166ac','#74add1','#fdae61','#d73027']
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].bar(level_order, summary['success_rate'],    color=colors)
axes[0].set_title('Success rate [%]'); axes[0].set_ylim(0,105); axes[0].set_ylabel('%')

axes[1].bar(level_order, summary['mean_min_psi'],    color=colors)
axes[1].axhline(0, color='red', lw=1, ls='--')
axes[1].set_title('Mean min safety margin \u03c8(x)')

axes[2].bar(level_order, summary['mean_violations'], color=colors)
axes[2].set_title('Mean # safety violations (\u03c8<0)')

for ax in axes:
    ax.set_xlabel('Disturbance level'); ax.grid(axis='y', alpha=0.3)

fig.suptitle('Vanilla MPC — Robustness (Dubins car)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('fig_robustness_vanilla_mpc.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nSummary:')
print(summary.round(3).to_string())